🗂️ Notebook Outline  
│  
├── 1. Imports, Setup & Configuration  
├── 2. Data Loading  
├── 3. Evaluation Functions (map@k, soft_map@k)  
├── 4. Retrieval Pipelines  
│   ├── 4.1 BM25  
│   ├── 4.2 SBERT (bi-encoder)  
│   └── 4.3 BM25 + Cross-Encoder (reranking)  
├── 5. Evaluation Runners  
│   ├── run_bm25_retrieval()  
│   ├── run_sbert_retrieval()  
│   └── run_cross_encoder_retrieval()  
├── 6. Results Comparison  
└── 7. Summary & Observations  



In [ ]:
#  BM25 Retrieval
def run_bm25_retrieval(query_df, product_df, bm25, top_k=10):
    def bm25_search(query):
        tokenized_query = word_tokenize(query.lower())
        scores = bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[-top_k:][::-1]
        return product_df.iloc[top_indices]['product_id'].tolist()

    query_df['top_product_ids'] = query_df['query'].apply(bm25_search)
    return query_df


In [ ]:
# SBERT Retrieval
def run_sbert_retrieval(query_df, product_df, sbert_model, product_embeddings, top_k=10):
    def sbert_search(query):
        query_embedding = sbert_model.encode(query, convert_to_tensor=True)
        scores = util.pytorch_cos_sim(query_embedding, product_embeddings)[0].cpu().numpy()
        top_indices = np.argsort(scores)[-top_k:][::-1]
        return product_df.iloc[top_indices]['product_id'].tolist()

    query_df['top_product_ids'] = query_df['query'].apply(sbert_search)
    return query_df


In [ ]:
# BM25 + Cross-Encoder Retrieval
def run_cross_encoder_retrieval(query_df, product_df, bm25, cross_encoder, top_k=10, top_n=50):
    def cross_encoder_search(query):
        tokenized_query = word_tokenize(query.lower())
        scores = bm25.get_scores(tokenized_query)
        candidate_indices = np.argsort(scores)[-top_n:][::-1]
        candidates = product_df.iloc[candidate_indices]

        texts = (candidates['product_name'] + ' ' + candidates['product_description']).fillna('').tolist()
        pairs = [(query, doc) for doc in texts]
        rerank_scores = cross_encoder.predict(pairs, show_progress_bar=False)
        reranked = np.argsort(rerank_scores)[::-1][:top_k]
        return product_df.iloc[[candidate_indices[i] for i in reranked]]['product_id'].tolist()

    query_df['top_product_ids'] = query_df['query'].apply(cross_encoder_search)
    return query_df


In [ ]:
# evaluation call:

def evaluate_retrieval(query_df, label_df, k=10, use_soft=False):
    grouped_label_df = label_df.groupby('query_id')

    def get_exact(query_id):
        return grouped_label_df.get_group(query_id).loc[lambda df: df['label'] == 'Exact']['product_id'].values

    def get_partial(query_id):
        return grouped_label_df.get_group(query_id).loc[lambda df: df['label'] == 'Partial']['product_id'].values

    query_df['relevant_ids'] = query_df['query_id'].apply(get_exact)
    if use_soft:
        query_df['partial_ids'] = query_df['query_id'].apply(get_partial)
        query_df['soft_map@k'] = query_df.apply(lambda x: soft_map_at_k(x['relevant_ids'], x['top_product_ids'], x['partial_ids'], k=k), axis=1)
        return query_df['soft_map@k'].mean()
    else:
        query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=k), axis=1)
        return query_df['map@k'].mean()


In [ ]:
bm25_score = evaluate_retrieval(run_bm25_retrieval(query_df.copy(), product_df, bm25), label_df)
sbert_score = evaluate_retrieval(run_sbert_retrieval(query_df.copy(), product_df, sbert_model, product_embeddings), label_df)
cross_score = evaluate_retrieval(run_cross_encoder_retrieval(query_df.copy(), product_df, bm25, cross_encoder), label_df)

print(f"BM25 MAP@10:        {bm25_score:.4f}")
print(f"SBERT MAP@10:       {sbert_score:.4f}")
print(f"Cross-Encoder MAP@10: {cross_score:.4f}")
